# Historical analysis — public source copy
Outputs and execution counts were cleared for publication privacy. Original evidence remains local.
Do not Run All modeling notebooks, retrain frozen models or reevaluate the consumed test.
Dataset/model assets are withheld; see the release README.

# 01 — Data understanding

Phase 1: acquire and inspect the original IBM Telco Customer Churn data. No cleaning, EDA conclusions, model fitting, or dashboard is implemented. The source is a customer snapshot; associations cannot establish causality or time trends.

In [ ]:
from pathlib import Path
import sys
import json

# Works when launched from the project root or notebooks directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "src/churn_analytics").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/churn_analytics").is_dir(), "Launch from the project root or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
from churn_analytics.data import download_dataset, load_dataset, validate_dataset
print("Python executable:", sys.executable)


## Acquire and load

The downloader reuses an existing file and preserves raw data. If offline, follow the exception’s manual-download instructions and rerun. The expected original size is 7,043 rows × 21 columns. Customer IDs identify records; they must never become predictive features.

In [ ]:
raw_path = ROOT / "data/raw/Telco-Customer-Churn.csv"
download_dataset(raw_path)
df = load_dataset(raw_path)
print("Shape:", df.shape)
df.head()

## Validate before analysis

Check the exact schema, inferred types, nulls, whitespace, duplicate and missing IDs, numeric parse failures, and Yes/No target counts. A row-count difference is a warning to investigate. Schema, ID, target, and unexpected numeric parsing problems are errors. pandas types are inferred representations, not proof that values have the correct business meaning.

In [ ]:
report = validate_dataset(df)
print(json.dumps(report, indent=2))
assert not report["errors"], report["errors"]

## Investigate TotalCharges

Whitespace-only strings are not counted by `isna()`. They can make a numeric-looking column load as object/string. Convert only a temporary diagnostic series and examine tenure for these records. Do not drop rows or fill charges yet. Even if every blank belongs to tenure zero, that observation alone does not establish the correct financial value.

In [ ]:
blank = df["TotalCharges"].astype("string").str.fullmatch(r"\s*").fillna(False)
print(report["total_charges_investigation"])
df.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

## Review questions and handoff

- Does the file match 7,043 × 21 and the expected column names?
- Are customer IDs unique and complete?
- Do target counts contain only Yes and No?
- How do whitespace and null counts differ?
- Do all blank TotalCharges records have zero tenure?

Phase 2 will choose and document a cleaning policy and begin EDA after explicit authorization. Future modeling must split before fitting preprocessing, exclude IDs, use pipelines, and reserve the final test set for final evaluation.